# **매장·상품군 수요 패턴 군집**

전체 정확도만 보면 **어떤 수요에서 예측이 쓸모 있는지** 알기 어렵습니다. 매장×상품군 시계열을 판매 규모, 0 판매 비율, 변동성, 요일성 등으로 묶고, 각 유형의 특징을 설명합니다. 개별 상품이나 고객의 구매 순서를 분석하는 것은 아닙니다.

군집은 Valid 시작 전인 **2017-07-18까지의 180일**로 정의합니다. 이후 판매량은 군집 생성에 넣지 않으며, [08 평가](08_model_evaluation_and_decision.ipynb)에서 유형별 예측 오류를 읽는 데 사용합니다.

## **환경 준비**

별도의 00 노트북에서 저장소를 My Drive에 clone·갱신한 뒤 실행합니다. Drive를 연결하고 `/content/drive/MyDrive/retail-demand-forecasting`을 작업 폴더로 사용합니다.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive/")
except ImportError:
    pass

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

## **패키지 준비**

Colab에서는 먼저 별도의 00 노트북으로 Drive를 연결하고 저장소와 Favorita `train.csv`를 준비합니다. 로컬에서는 저장소 루트 또는 `notebooks` 폴더에서 실행할 수 있습니다. 아래 패키지는 기존 `requirements.txt`에 포함되어 있습니다.

In [ ]:
%pip install -q pandas pyarrow scikit-learn seaborn koreanize-matplotlib
%pip install -q -e . --no-deps

## **Library**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

from retail_forecast.segments import (
    CLUSTER_COLUMNS,
    build_segment_features,
    fit_demand_segments,
    load_segment_history,
    summarize_segments,
)

## **Utils**

대용량 원본 CSV의 기간 선택과 군집 Feature 계산은 `src/retail_forecast/segments.py`에 있습니다. 이 노트북은 설정·시각화·해석만 담당합니다.

## **분석 설정**

In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

CUTOFF = pd.Timestamp("2017-07-18")
HISTORY_DAYS = 180
N_CLUSTERS = 4
MIN_COVERAGE = 0.90

RAW_PATH = REPO / "data" / "raw"
OUTPUT_DIR = REPO / "outputs" / "demand_segments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert (RAW_PATH / "train.csv").is_file(), RAW_PATH / "train.csv"

## **1. 학습 기간의 수요 이력**

`train.csv`를 필요한 열만 청크로 읽고 최근 180일만 남깁니다. 0 판매는 관측값으로 보존합니다. 행 자체가 없는 날은 `coverage_ratio`에 반영합니다.

In [ ]:
history = load_segment_history(RAW_PATH, CUTOFF, HISTORY_DAYS)
history_overview = pd.DataFrame([{
    "시작일": history["date"].min().date(),
    "마지막일": history["date"].max().date(),
    "관측 행": len(history),
    "매장·상품군": history[["store_nbr", "family"]].drop_duplicates().shape[0],
    "0 판매 비율": history["sales"].eq(0).mean(),
}])
history_overview

## **2. 시계열별 요약과 군집**

평균 판매량은 `log1p`로 규모를 완화합니다. 판매 변동과 요일별 평균의 변동은 표준편차/평균을 요약한 뒤 로그 변환합니다. 추세는 마지막 28일과 처음 28일의 평균 판매량 차이를 로그 규모에서 비교합니다. 판촉일 비율은 설명 변수이며 **판촉의 인과효과**를 뜻하지 않습니다.

4개 군집은 해석을 위한 고정 설정입니다. 번호 `0`~`3`에는 크기나 품질의 순서가 없으며, 아래 요약표를 보고 의미를 붙입니다. Silhouette 값은 군집 분리 정도의 참고값으로만 봅니다.

### **군집 분리 정도와 이력 부족 여부**


In [ ]:
segment_features = build_segment_features(history, CUTOFF, HISTORY_DAYS)
segment_features, silhouette = fit_demand_segments(
    segment_features, clusters=N_CLUSTERS, min_coverage=MIN_COVERAGE,
)
profile = summarize_segments(segment_features)
segment_features["history_cutoff"] = CUTOFF

segment_features.to_parquet(OUTPUT_DIR / "segment_features.parquet", index=False)
profile.to_csv(OUTPUT_DIR / "segment_profile.csv", index=False)
segment_quality = pd.DataFrame([{"Silhouette": silhouette, "이력 부족 시계열":
                       segment_features["segment"].eq("이력 부족").sum()}])
segment_quality


### **군집별 전형적인 판매 패턴**


In [ ]:
profile_view = profile.round(3)
profile_view


## **3. 수요 패턴 시각화**

왼쪽 그림은 시계열별 판매 규모와 0 판매 비율을 보여줍니다. Heatmap은 각 Feature의 전체 시계열 평균 대비 차이를 표준편차 단위로 표시합니다. 실제 단위와 비율은 위 요약표에서 함께 확인합니다.

In [ ]:
clustered = segment_features.loc[segment_features["segment"].ne("이력 부족")]
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=clustered, x="log_mean_sales", y="zero_share", hue="segment",
                alpha=0.65, s=35, ax=ax)
ax.set(xlabel="log(1 + 일평균 판매량)", ylabel="0 판매 비율", title="매장·상품군 수요 유형")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "segment_scatter.png", dpi=160)
plt.show()

In [ ]:
feature_medians = clustered.groupby("segment", observed=True)[CLUSTER_COLUMNS].median()
feature_scale = clustered[CLUSTER_COLUMNS].std(ddof=0).replace(0, 1)
relative_profile = (feature_medians - clustered[CLUSTER_COLUMNS].mean()) / feature_scale

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(relative_profile.T, annot=True, fmt=".1f", cmap="vlag", center=0, ax=ax)
ax.set(xlabel="군집", ylabel="", title="전체 시계열 평균 대비 군집 특징 (표준편차 단위)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "segment_profile_heatmap.png", dpi=160)
plt.show()

## **결과를 읽는 순서**

1. 각 군집의 시계열 수와 판매 규모를 확인합니다. 작은 군집이나 `이력 부족`은 별도로 표시합니다.
2. 0 판매 비율·요일성·변동성·추세로 **패턴을 설명하는 이름**을 붙입니다. 번호 자체에는 의미가 없습니다.
3. [08 평가 노트북](08_model_evaluation_and_decision.ipynb)의 군집별 7일 WAPE·Bias·판매량 비중을 확인합니다. 02를 실행한 뒤 08의 분석 셀을 다시 실행하면 됩니다. 모델 학습은 필요하지 않습니다.

이 군집은 현재 한 시점의 기술적 요약입니다. 시점이 바뀌면 군집 경계와 소속도 바뀔 수 있으므로 운영 규칙으로 바로 고정하지 않습니다.